In [2]:
import sys
import sqlite3
import hashlib
import uuid
from pathlib import Path

# Configurar path del proyecto
PROJECT_ROOT = Path(r'D:\Master\TrabajoFinalUCM\TFM')
sys.path.insert(0, str(PROJECT_ROOT))

DB1_PATH = str(PROJECT_ROOT / 'data' / 'tui_recomendador.db')
DB2_PATH = str(PROJECT_ROOT / 'data' / 'tui_recomendador_javier_v2.db')

print(f'DB1: {DB1_PATH}')
print(f'DB2: {DB2_PATH}')

DB1: D:\Master\TrabajoFinalUCM\TFM\data\tui_recomendador.db
DB2: D:\Master\TrabajoFinalUCM\TFM\data\tui_recomendador_javier_v2.db


# Análisis de tui_recomendador.db

In [3]:
conn1 = sqlite3.connect(DB1_PATH)

total1 = conn1.execute('SELECT COUNT(*) FROM resenas').fetchone()[0]
print(f'Total registros: {total1}')

print('\nPor fuente:')
for f, n in conn1.execute('SELECT fuente, COUNT(*) FROM resenas GROUP BY fuente ORDER BY COUNT(*) DESC').fetchall():
    print(f'  {f}: {n}')

print('\nTop 15 destinos:')
for d, n in conn1.execute('SELECT destino_nombre, COUNT(*) FROM resenas GROUP BY destino_nombre ORDER BY COUNT(*) DESC LIMIT 15').fetchall():
    print(f'  {d}: {n}')

print('\nPor idioma:')
for i, n in conn1.execute('SELECT idioma, COUNT(*) FROM resenas GROUP BY idioma ORDER BY COUNT(*) DESC LIMIT 10').fetchall():
    print(f'  {i}: {n}')

conn1.close()

Total registros: 20117

Por fuente:
  reddit: 11649
  youtube: 6841
  tripadvisor: 1621
  google_maps: 6

Top 15 destinos:
  Punta Cana: 278
  Bali: 264
  Tenerife: 243
  Mallorca: 241
  Ibiza: 236
  Santorini: 235
  Jamaica: 225
  Riviera Maya: 224
  Antalya: 222
  Cuba: 221
  Costa Rica: 220
  Cancun: 216
  Hurghada: 208
  Algarve: 205
  Sri Lanka: 194

Por idioma:
  en: 10243
  es: 8305
  pt: 327
  it: 267
  ca: 212
  fr: 163
  de: 147
  id: 57
  tl: 43
  ro: 36


# Análisis de tui_recomendador de javier

In [4]:
conn2 = sqlite3.connect(DB2_PATH)

total2 = conn2.execute('SELECT COUNT(*) FROM resenas').fetchone()[0]
print(f'Total registros: {total2}')

print('\nPor fuente:')
for f, n in conn2.execute('SELECT fuente, COUNT(*) FROM resenas GROUP BY fuente ORDER BY COUNT(*) DESC').fetchall():
    print(f'  {f}: {n}')

print('\nTop 15 destinos:')
for d, n in conn2.execute('SELECT destino_nombre, COUNT(*) FROM resenas GROUP BY destino_nombre ORDER BY COUNT(*) DESC LIMIT 15').fetchall():
    print(f'  {d}: {n}')

print('\nPor idioma:')
for i, n in conn2.execute('SELECT idioma, COUNT(*) FROM resenas GROUP BY idioma ORDER BY COUNT(*) DESC LIMIT 10').fetchall():
    print(f'  {i}: {n}')

conn2.close()

Total registros: 30639

Por fuente:
  reddit: 17464
  youtube: 10425
  tripadvisor: 2748
  google_maps: 2

Top 15 destinos:
  Jamaica: 219
  Ibiza: 203
  Bali: 202
  Costa Rica: 199
  Tenerife: 189
  Riviera Maya: 188
  Santorini: 187
  Las Vegas: 185
  Sri Lanka: 181
  Mallorca: 181
  Algarve: 180
  Gran Canaria: 177
  Antalya: 174
  Punta Cana: 173
  Bulgaria: 172

Por idioma:
  en: 14302
  es: 13140
  pt: 467
  de: 417
  fr: 387
  it: 363
  ca: 301
  tr: 283
  nl: 121
  pl: 85


# . Cruce y comparación 

In [5]:
# Calcular hashes de ambas BDs
def get_hashes(db_path):
    conn = sqlite3.connect(db_path)
    textos = conn.execute('SELECT texto_original FROM resenas WHERE texto_original IS NOT NULL').fetchall()
    hashes = set()
    for (t,) in textos:
        if t and len(t.strip()) > 10:
            hashes.add(hashlib.md5(t.strip().lower().encode()).hexdigest())
    conn.close()
    return hashes

hashes1 = get_hashes(DB1_PATH)
hashes2 = get_hashes(DB2_PATH)

comunes = hashes1 & hashes2
solo_db1 = hashes1 - hashes2
solo_db2 = hashes2 - hashes1
union = hashes1 | hashes2

print(f'Únicos en tui_recomendador.db:        {len(hashes1)}')
print(f'Únicos en tui_recomendador-javier.db: {len(hashes2)}')
print(f'En COMÚN (duplicados):                {len(comunes)}')
print(f'Solo en tui_recomendador.db:          {len(solo_db1)}')
print(f'Solo en javier.db:                    {len(solo_db2)}')
print(f'UNIÓN (total sin duplicados):         {len(union)}')
print(f'% de cruce:                           {len(comunes)/max(1,min(len(hashes1),len(hashes2)))*100:.1f}%')

print(f'\n--- RESUMEN ---')
print(f'Si unificamos: {len(union)} reseñas únicas')
print(f'Se ganarían {len(solo_db2)} reseñas nuevas de javier.db')
print(f'Se eliminarían {len(comunes)} duplicados')

Únicos en tui_recomendador.db:        20117
Únicos en tui_recomendador-javier.db: 30636
En COMÚN (duplicados):                13926
Solo en tui_recomendador.db:          6191
Solo en javier.db:                    16710
UNIÓN (total sin duplicados):         36827
% de cruce:                           69.2%

--- RESUMEN ---
Si unificamos: 36827 reseñas únicas
Se ganarían 16710 reseñas nuevas de javier.db
Se eliminarían 13926 duplicados


# Unificar left jin 

In [6]:
# Ejecuta esta celda SOLO si quieres unificar

conn1 = sqlite3.connect(DB1_PATH)
conn2 = sqlite3.connect(DB2_PATH)

# Cargar hashes existentes en DB1
textos1 = conn1.execute('SELECT texto_original FROM resenas WHERE texto_original IS NOT NULL').fetchall()
hashes_db1 = set()
for (t,) in textos1:
    if t and len(t.strip()) > 10:
        hashes_db1.add(hashlib.md5(t.strip().lower().encode()).hexdigest())

# Leer reseñas de DB2
resenas2 = conn2.execute("""
    SELECT destino_nombre, fuente, texto_original, idioma, puntuacion,
           fecha_publicacion, url_fuente, fecha_extraccion
    FROM resenas WHERE texto_original IS NOT NULL
""").fetchall()

# Insertar solo las nuevas
insertadas = 0
for row in resenas2:
    destino, fuente, texto, idioma, puntuacion, fecha_pub, url, fecha_ext = row
    if not texto or len(texto.strip()) <= 10:
        continue
    h = hashlib.md5(texto.strip().lower().encode()).hexdigest()
    if h in hashes_db1:
        continue
    hashes_db1.add(h)
    conn1.execute(
        """INSERT INTO resenas (id_resena, destino_nombre, fuente, texto_original,
           idioma, puntuacion, fecha_publicacion, url_fuente, fecha_extraccion)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        (str(uuid.uuid4()), destino, fuente, texto, idioma, puntuacion, fecha_pub, url, fecha_ext)
    )
    insertadas += 1

conn1.commit()
total_final = conn1.execute('SELECT COUNT(*) FROM resenas').fetchone()[0]
conn1.close()
conn2.close()

print(f'Unificación completada')
print(f'   Reseñas nuevas insertadas: {insertadas}')
print(f'   Total final en tui_recomendador.db: {total_final}')

Unificación completada
   Reseñas nuevas insertadas: 16710
   Total final en tui_recomendador.db: 36827
